In [0]:
source_path = '/Volumes/finguard/source/fraud_watchlist/source_data/'





In [0]:
input_stream = (
    spark.readStream.format('cloudFiles')
    .option('cloudFiles.format', 'json')
    .option('cloudFiles.schemaLocation', '/Volumes/finguard/source/fraud_watchlist/schema')
    .option('cloudFiles.inferColumnTypes', True)
    .load(source_path)
)

In [0]:
from pyspark.sql import functions as F

In [0]:
transformed_df = input_stream.select(
    "*",
    F.col("_metadata.file_path").alias("file_path"),
    F.current_timestamp().alias("ingest_timestamp")
)

In [0]:
streaming_query=(transformed_df.writeStream.format("delta")
.outputMode("Append")
.option("checkpointLocation", "/Volumes/finguard/source/fraud_watchlist/checkpoint/")
.trigger(availableNow=True)
.toTable("finguard.bronze.fraud_transactions_watch_batch_list")
)